# TakeMeter Training Notebook

Fine-tune DistilBERT on r/nba discourse quality labels and compare to a zero-shot Groq baseline.

**Sections:**
1. Label map + CSV upload
2. Dataset split and tokenization
3. Fine-tuning
4. Fine-tuned model evaluation
5. Zero-shot baseline (Groq)

> **Before running:** Set runtime to T4 GPU via Runtime → Change runtime type → T4 GPU.

In [ ]:
# Install dependencies (pre-installed on Colab, but explicit for reproducibility)
!pip install -q transformers datasets scikit-learn groq

---
## Section 1: Label Map and Data Upload

Define your label map and upload your labeled CSV. The notebook expects a CSV with at minimum:
- `text` — the post or comment
- `label` — the string label (must match keys in `LABEL_MAP` below)

In [ ]:
# ─── CONFIGURE YOUR LABELS HERE ───────────────────────────────────────────────
# Keys must exactly match the string labels in your CSV.
# Values are integer IDs used internally by the model.
LABEL_MAP = {
    "analysis": 0,
    "hot_take": 1,
    "reaction": 2,
}
ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
NUM_LABELS = len(LABEL_MAP)

print(f"Labels ({NUM_LABELS}): {list(LABEL_MAP.keys())}")

In [ ]:
import pandas as pd
from google.colab import files

print("Upload your labeled CSV file (must have 'text' and 'label' columns):")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)
print(f"\nLoaded {len(df)} examples from '{filename}'")
print(f"Columns: {list(df.columns)}")

# Validate required columns
assert "text" in df.columns, "CSV must have a 'text' column"
assert "label" in df.columns, "CSV must have a 'label' column"

# Drop rows with missing text or label
df = df.dropna(subset=["text", "label"])
df["text"] = df["text"].astype(str).str.strip()

# Validate that all labels are known
unknown = set(df["label"].unique()) - set(LABEL_MAP.keys())
if unknown:
    raise ValueError(f"Unknown labels in CSV: {unknown}. Expected: {list(LABEL_MAP.keys())}")

# Convert string labels to integer IDs
df["label_id"] = df["label"].map(LABEL_MAP)

print("\nLabel distribution:")
print(df["label"].value_counts().to_string())
print(f"\nClass balance:")
for label, count in df["label"].value_counts().items():
    pct = count / len(df) * 100
    flag = " ⚠️  (>70% — consider rebalancing)" if pct > 70 else ""
    print(f"  {label}: {count} ({pct:.1f}%){flag}")

---
## Section 2: Dataset Split and Tokenization

Splits the dataset 70% train / 15% validation / 15% test and tokenizes all splits.

> **Important:** The test set is locked here and used for both the fine-tuned model and the baseline. Do not look at test set performance until you have finished training.

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
RANDOM_SEED = 42

# Split: 70 / 15 / 15
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=RANDOM_SEED, stratify=df["label_id"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=RANDOM_SEED, stratify=temp_df["label_id"]
)

print(f"Split sizes — Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print("\nTrain label distribution:")
print(train_df["label"].value_counts().to_string())
print("\nTest label distribution:")
print(test_df["label"].value_counts().to_string())

# Tokenize
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def df_to_dataset(df):
    ds = Dataset.from_dict({"text": df["text"].tolist(), "labels": df["label_id"].tolist()})
    return ds.map(tokenize, batched=True)

train_ds = df_to_dataset(train_df)
val_ds   = df_to_dataset(val_df)
test_ds  = df_to_dataset(test_df)

# Store test texts and true labels for Section 4 and Section 5
test_texts  = test_df["text"].tolist()
test_labels = test_df["label"].tolist()
test_ids    = test_df["label_id"].tolist()

print(f"\nTokenization complete. Max sequence length: {MAX_LENGTH}")

---
## Section 3: Fine-Tuning

Fine-tunes `distilbert-base-uncased` on your training data.

**Default hyperparameters:**
- Epochs: 4
- Learning rate: 2e-5
- Batch size: 16

Training takes approximately 5–15 minutes on a T4 GPU with 200 examples.

In [ ]:
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# ─── HYPERPARAMETERS ──────────────────────────────────────────────────────────
NUM_EPOCHS    = 4
LEARNING_RATE = 2e-5
BATCH_SIZE    = 16
WARMUP_STEPS  = 50
# ──────────────────────────────────────────────────────────────────────────────

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL,
    label2id=LABEL_MAP,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "macro_f1": f1}

training_args = TrainingArguments(
    output_dir="./takemeter_model",
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    warmup_steps=WARMUP_STEPS,
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print(f"Starting training: {NUM_EPOCHS} epochs, lr={LEARNING_RATE}, batch={BATCH_SIZE}")
trainer.train()
print("Training complete.")

---
## Section 4: Fine-Tuned Model Evaluation

Evaluates the fine-tuned model on the locked test set, prints per-class metrics, generates a confusion matrix, and downloads `evaluation_results.json` and `confusion_matrix.png`.

In [ ]:
import json
import torch
import matplotlib.pyplot as plt
import matplotlib
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
)

# ── Run inference on test set ──────────────────────────────────────────────────
predictions_output = trainer.predict(test_ds)
logits = predictions_output.predictions
probs  = torch.softmax(torch.tensor(logits), dim=-1).numpy()
preds  = np.argmax(logits, axis=-1)

# ── Overall accuracy ───────────────────────────────────────────────────────────
acc = accuracy_score(test_ids, preds)
print(f"Fine-tuned model — Test accuracy: {acc:.4f} ({acc*100:.1f}%)\n")

# ── Per-class report ───────────────────────────────────────────────────────────
label_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
report = classification_report(test_ids, preds, target_names=label_names, output_dict=True)
report_str = classification_report(test_ids, preds, target_names=label_names)
print(report_str)

# ── Confusion matrix ───────────────────────────────────────────────────────────
cm = confusion_matrix(test_ids, preds)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Fine-Tuned DistilBERT — Confusion Matrix (Test Set)", fontsize=13)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("Saved: confusion_matrix.png")

# ── Wrong predictions ──────────────────────────────────────────────────────────
wrong = [
    {
        "text": test_texts[i],
        "true_label": ID_TO_LABEL[test_ids[i]],
        "predicted_label": ID_TO_LABEL[preds[i]],
        "confidence": float(probs[i][preds[i]]),
    }
    for i in range(len(test_texts))
    if preds[i] != test_ids[i]
]
print(f"\nWrong predictions ({len(wrong)} total):")
for w in wrong:
    print(f"  TRUE={w['true_label']:10s}  PRED={w['predicted_label']:10s}  CONF={w['confidence']:.2f}  '{w['text'][:70]}...'")

# ── Sample classifications (5 correct examples with confidence) ────────────────
samples = [
    {
        "text": test_texts[i],
        "predicted_label": ID_TO_LABEL[preds[i]],
        "confidence": float(probs[i][preds[i]]),
        "correct": bool(preds[i] == test_ids[i]),
    }
    for i in range(len(test_texts))
    if preds[i] == test_ids[i]
][:5]

print("\nSample classifications (correct predictions):")
for s in samples:
    print(f"  PRED={s['predicted_label']:10s}  CONF={s['confidence']:.2f}  '{s['text'][:70]}'")

# ── Save evaluation results ────────────────────────────────────────────────────
results = {
    "fine_tuned_model": {
        "overall_accuracy": round(acc, 4),
        "per_class": {
            label: {
                "precision": round(report[label]["precision"], 3),
                "recall":    round(report[label]["recall"], 3),
                "f1":        round(report[label]["f1-score"], 3),
                "support":   report[label]["support"],
            }
            for label in label_names
        },
        "macro_avg": {
            "precision": round(report["macro avg"]["precision"], 3),
            "recall":    round(report["macro avg"]["recall"], 3),
            "f1":        round(report["macro avg"]["f1-score"], 3),
        },
        "confusion_matrix": {
            "labels": label_names,
            "matrix": cm.tolist(),
        },
    },
    "wrong_predictions": wrong,
    "sample_classifications": samples,
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved: evaluation_results.json")

In [ ]:
# Download output files to your local machine
from google.colab import files
files.download("confusion_matrix.png")
files.download("evaluation_results.json")
print("Downloads started. Add these files to your GitHub repo under outputs/.")

---
## Section 5: Zero-Shot Baseline (Groq)

Classifies every example in the locked test set using `llama-3.3-70b-versatile` via Groq — no fine-tuning, just your label definitions in the prompt.

**Setup:**
1. Click the 🔑 icon in the left Colab sidebar
2. Add a secret named `GROQ_API_KEY` with your Groq API key
3. Enable notebook access for the secret

**Sections 1 and 2 must have been run before this section.** If your runtime reset, re-upload the CSV and re-run Sections 1 and 2.

In [ ]:
import os
import time
from groq import Groq
from google.colab import userdata
from sklearn.metrics import classification_report, accuracy_score

# ── Groq client ────────────────────────────────────────────────────────────────
# Load key from Colab Secrets (recommended). Alternatively, paste your key directly:
# GROQ_API_KEY = "gsk_..."
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)

BASELINE_MODEL = "llama-3.3-70b-versatile"

# ── Classification prompt ──────────────────────────────────────────────────────
# Edit this prompt to match your label definitions exactly as written in planning.md.
SYSTEM_PROMPT = """You are classifying posts from the r/nba subreddit into one of three categories.

- analysis: a structured argument backed by statistics, historical comparison, or tactical observation. Evidence is specific and verifiable.
- hot_take: a bold confident opinion stated without supporting evidence. The claim may be true, but the post asserts rather than argues.
- reaction: an immediate emotional response to a specific game, play, or event. Little to no argument — expressing a feeling in the moment.

Respond with ONLY one of these exact strings: analysis, hot_take, or reaction. No punctuation, no explanation."""

def classify_baseline(text: str) -> str | None:
    """Returns predicted label string, or None if response is unparseable."""
    try:
        response = client.chat.completions.create(
            model=BASELINE_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Post: {text}"},
            ],
            temperature=0,
            max_tokens=10,
        )
        raw = response.choices[0].message.content.strip().lower()
        # Accept the response only if it exactly matches a known label
        if raw in LABEL_MAP:
            return raw
        # Try to find a label anywhere in the response (fallback)
        for label in LABEL_MAP:
            if label in raw:
                return label
        return None
    except Exception as e:
        print(f"  API error: {e}")
        return None

# ── Run baseline on test set ────────────────────────────────────────────────────
print(f"Running zero-shot baseline on {len(test_texts)} test examples...")
baseline_preds = []
unparseable = []

for i, text in enumerate(test_texts):
    pred = classify_baseline(text)
    baseline_preds.append(pred)
    if pred is None:
        unparseable.append(i)
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(test_texts)} classified...")
    time.sleep(0.3)  # Avoid rate limiting

print(f"\nComplete. Unparseable responses: {len(unparseable)}")
if len(unparseable) / len(test_texts) > 0.10:
    print("⚠️  >10% unparseable — revise your prompt to produce cleaner output.")

In [ ]:
# Evaluate baseline — skip unparseable examples
valid_idx = [i for i, p in enumerate(baseline_preds) if p is not None]
valid_true  = [test_labels[i] for i in valid_idx]
valid_preds = [baseline_preds[i] for i in valid_idx]

baseline_acc = accuracy_score(valid_true, valid_preds)
print(f"Zero-shot baseline — Accuracy: {baseline_acc:.4f} ({baseline_acc*100:.1f}%)")
print(f"(Evaluated on {len(valid_idx)}/{len(test_texts)} parseable responses)\n")

print(classification_report(valid_true, valid_preds, target_names=label_names))

# ── Update evaluation_results.json with baseline numbers ──────────────────────
baseline_report = classification_report(
    valid_true, valid_preds, target_names=label_names, output_dict=True
)

with open("evaluation_results.json", "r") as f:
    results = json.load(f)

results["baseline_model"] = {
    "name": f"{BASELINE_MODEL} (zero-shot via Groq)",
    "overall_accuracy": round(baseline_acc, 4),
    "per_class": {
        label: {
            "precision": round(baseline_report[label]["precision"], 3),
            "recall":    round(baseline_report[label]["recall"], 3),
            "f1":        round(baseline_report[label]["f1-score"], 3),
            "support":   baseline_report[label]["support"],
        }
        for label in label_names
    },
    "macro_avg": {
        "precision": round(baseline_report["macro avg"]["precision"], 3),
        "recall":    round(baseline_report["macro avg"]["recall"], 3),
        "f1":        round(baseline_report["macro avg"]["f1-score"], 3),
    },
    "unparseable_responses": len(unparseable),
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Updated evaluation_results.json with baseline results.")
print("\n--- Summary Comparison ---")
print(f"Zero-shot baseline accuracy : {baseline_acc*100:.1f}%")
print(f"Fine-tuned model accuracy   : {results['fine_tuned_model']['overall_accuracy']*100:.1f}%")
improvement = (results['fine_tuned_model']['overall_accuracy'] - baseline_acc) * 100
print(f"Improvement from fine-tuning: +{improvement:.1f} percentage points")

In [ ]:
# Download the updated evaluation_results.json with baseline numbers
from google.colab import files
files.download("evaluation_results.json")
print("Download started. Replace outputs/evaluation_results.json in your repo.")

---
## Optional: Classify a New Post

Run inference on a single post using the fine-tuned model to see label + confidence.

In [ ]:
def classify_post(text: str):
    """Run the fine-tuned model on a single post and return label + confidence."""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    pred_id = probs.argmax().item()
    label = ID_TO_LABEL[pred_id]
    confidence = probs[pred_id].item()
    print(f"Text     : {text[:100]}")
    print(f"Label    : {label}")
    print(f"Confidence: {confidence:.2%}")
    print("All scores:")
    for i, p in enumerate(probs):
        print(f"  {ID_TO_LABEL[i]:12s}: {p.item():.2%}")
    return label, confidence

# Try it:
classify_post("Haliburton's clutch time TS% this playoffs (58.2%) is top 5 among all players with 20+ clutch possessions. The sample is real.")

---
## Export Model for Local Use

Run this cell after training to download the model so you can run the React demo locally.

In [ ]:
import shutil
from google.colab import files

# Save the best model + tokenizer to a single folder
model.save_pretrained("./takemeter_export")
tokenizer.save_pretrained("./takemeter_export")

# Zip and download
shutil.make_archive("takemeter_model", "zip", "./takemeter_export")
files.download("takemeter_model.zip")
print("Downloaded takemeter_model.zip — unzip into app/backend/takemeter_model/")